In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings

In [2]:

# Turn off unnecessary Pandas warnings
warnings.filterwarnings('ignore')

# --- 1. DETERMINING FILE PATHS ---
input_dir = "../02_Processed_Data/02_Necessary_Columns"
output_dir = "../02_Processed_Data/03_Calculations_Features"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Find input files
file_paths = glob.glob(os.path.join(input_dir, '*.csv'))

# --- 2. COLUMN NAMES ---
col_imotor = 'IMotor'
col_trq = 'Kafa_Act_Trq'
col_x = 'X_Accelerometer'
col_y = 'Y_Accelerometer'
col_z = 'Z_Accelerometer'

# --- 3. PROCESSING PARAMETERS ---
window_size = 10  # Window size (Take the first 10 rows)
step_size = 1     # Advance by every row

print(f"A total of {len(file_paths)} files will be processed...\n")

for file_path in file_paths:
    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}")
    
    df_raw = pd.read_csv(file_path)
    
    if len(df_raw) < window_size:
        print(f"  -> Warning: Not enough data in {file_name}! Skipped.")
        continue
        
    # --- DATA CLEANING (SECURITY STEP) ---
    # Protection against comma and dot confusion (for D files, etc.)
    numeric_columns = [col_imotor, col_trq, col_x, col_y, col_z]
    for col in numeric_columns:
        if col in df_raw.columns:
            # Convert commas to dots and turn everything non-numeric (letters, spaces, etc.) into NaN
            df_raw[col] = pd.to_numeric(df_raw[col].astype(str).str.replace(',', '.'), errors='coerce')
            
    # Delete corrupted or NaN rows and reset the index
    df_raw = df_raw.dropna(subset=[col_x, col_y, col_z]).reset_index(drop=True)

    if len(df_raw) < window_size:
        print(f"  -> Warning: Not enough data left in {file_name} after cleaning! Skipped.")
        continue

    # --- RESULTANT ACCELERATION (VibRes) CALCULATION ---
    # Vector sum of X, Y, and Z accelerations
    if all(col in df_raw.columns for col in [col_x, col_y, col_z]):
        df_raw['VibRes'] = np.sqrt(df_raw[col_x]**2 + df_raw[col_y]**2 + df_raw[col_z]**2)
    else:
        print(f"  -> Error: Acceleration axes are missing in {file_name}!")
        continue

    total_rows = len(df_raw)
    processed_windows = []
    
    # Get File ID
    file_id = df_raw['File_ID'].iloc[0] if 'File_ID' in df_raw.columns else 'Unknown'
    
    # --- 4. ROLLING WINDOW LOOP ---
    for start_idx in range(0, total_rows - window_size + 1, step_size):
        end_idx = start_idx + window_size
        window_df = df_raw.iloc[start_idx:end_idx]
            
        imotor_data = window_df[col_imotor].values
        z_accel_data = window_df[col_z].values
        vibres_data = window_df['VibRes'].values
        torque_data = window_df[col_trq].values
        
        # --- Desired Basic Calculations ---
        imotor_std = np.nanstd(imotor_data, ddof=1)
        z_accel_std = np.nanstd(z_accel_data, ddof=1)
        vibres_std = np.nanstd(vibres_data, ddof=1)
        trq_std = np.nanstd(torque_data, ddof=1)
        
        # --- High Frequency Noise (HF Noise) ---
        imotor_diff = np.diff(imotor_data)
        imotor_hf_noise = np.nanstd(imotor_diff, ddof=1) if len(imotor_diff) > 0 else 0
        
        z_accel_diff = np.diff(z_accel_data)
        z_accel_hf_noise = np.nanstd(z_accel_diff, ddof=1) if len(z_accel_diff) > 0 else 0

        trq_diff = np.diff(torque_data)
        
        # --- High Frequency RMS (HighFreq RMS) ---
        z_highfreq_rms = np.sqrt(np.nanmean(z_accel_diff**2)) if len(z_accel_diff) > 0 else 0
        imotor_highfreq_rms = np.sqrt(np.nanmean(imotor_diff**2)) if len(imotor_diff) > 0 else 0
        trq_highfreq_rms = np.sqrt(np.nanmean(trq_diff**2)) if len(trq_diff) > 0 else 0
        
        # --- Cutting Progress Percentage (Index Based) ---
        # Ratio of the row where the window ends (end_idx) to total rows
        percent_complete = end_idx / total_rows

        # --- Adding Only Desired Columns to the Dictionary ---
        window_features = {
            'File_ID': file_id,
            'Percent_Complete': percent_complete,
            'IMotor_std': imotor_std,
            'Z_Accel_std': z_accel_std,
            'VibRes_Accel_std': vibres_std,
            'Trq_std': trq_std,
            'IMotor_HF_Noise': imotor_hf_noise,
            'Z_Accel_HF_Noise': z_accel_hf_noise,
            'Z_HighFreq_RMS': z_highfreq_rms,
            'IMotor_HighFreq_RMS': imotor_highfreq_rms,
            'Trq_HighFreq_RMS': trq_highfreq_rms
        }
        
        processed_windows.append(window_features)
    
    # --- 5. TRACKING CALCULATIONS ---
    if len(processed_windows) > 0:
        df_processed = pd.DataFrame(processed_windows)

        # Columns to track
        columns_to_track = ['IMotor_std', 'Z_Accel_std', 'VibRes_Accel_std', 'Trq_std','IMotor_HighFreq_RMS','Z_HighFreq_RMS']
        
        # PARAMETERS:
        dead_zone = 20  # Go back 20 steps (2 seconds) to avoid overlap with the current window
        past_reference_length = 50 # Take the average over 50 steps (5 seconds) (A more stable reference)

        for col in columns_to_track:
            # First, we shift(20) to prevent overlap. 
            # Then we take the average of the 50 windows backwards from that point.
            rolling_mean_past = df_processed[col].shift(dead_zone).rolling(window=past_reference_length, min_periods=1).mean()
            
            # The ratio of the current value to the completely independent average of the past 5 seconds
            ratio_column_name = f'{col}_Ratio_to_Past'
            df_processed[ratio_column_name] = np.where(rolling_mean_past > 0, 
                                                    df_processed[col] / rolling_mean_past, 
                                                    1.0) # If the past average is 0, let the ratio remain 1
            
            # Fill the initial NaNs caused by the shift with 1.0
            df_processed[ratio_column_name] = df_processed[ratio_column_name].fillna(1.0)

    
    # --- 6. TREND (SLOPE / ACCELERATION) CALCULATIONS (Smoothed Differencing) ---
        # We look at the rate of change (derivative) to understand if chatter is sneaking up (early warning).
        
        trend_columns = ['IMotor_std', 'Z_Accel_std', 'VibRes_Accel_std', 'Trq_std', 'IMotor_HighFreq_RMS', 'Z_HighFreq_RMS']
        
        for col in trend_columns:
            # Stage 1: To crush the instantaneous chip/burr noise and see the real trend, 
            # we take the moving average of the last 10 steps (1 second) (Smoothing).
            smoothed_signal = df_processed[col].rolling(window=10, min_periods=1).mean()
            
            # Stage 2: We take the MINUS difference of this smoothed signal with its state exactly 10 steps (1 second) ago.
            # Positive value: Vibration/Load is increasing (Heading towards chatter)
            # Zero: Stable cutting
            # Negative value: Vibration/Load is decreasing (Relief)
            trend_column_name = f'{col}_Trend'
            df_processed[trend_column_name] = smoothed_signal.diff(10).fillna(0.0)

        # --- SAVE THE FILE ---
        saved_filename = file_name.replace('nec_cols_', 'features_')
        out_filepath = os.path.join(output_dir, saved_filename)
        df_processed.to_csv(out_filepath, index=False)
        print(f"  -> Saved: {saved_filename} ({len(df_processed)} windows)")

A total of 20 files will be processed...

Processing: nec_cols_I.csv
  -> Saved: features_I.csv (1023 windows)
Processing: nec_cols_H.csv
  -> Saved: features_H.csv (1103 windows)
Processing: nec_cols_9.csv
  -> Saved: features_9.csv (1352 windows)
Processing: nec_cols_8.csv
  -> Saved: features_8.csv (1476 windows)
Processing: nec_cols_10.csv
  -> Saved: features_10.csv (1247 windows)
Processing: nec_cols_11.csv
  -> Saved: features_11.csv (1158 windows)
Processing: nec_cols_C.csv
  -> Saved: features_C.csv (1244 windows)
Processing: nec_cols_5.csv
  -> Saved: features_5.csv (1295 windows)
Processing: nec_cols_4.csv
  -> Saved: features_4.csv (1211 windows)
Processing: nec_cols_B.csv
  -> Saved: features_B.csv (1357 windows)
Processing: nec_cols_6.csv
  -> Saved: features_6.csv (1195 windows)
Processing: nec_cols_7.csv
  -> Saved: features_7.csv (1109 windows)
Processing: nec_cols_A.csv
  -> Saved: features_A.csv (1414 windows)
Processing: nec_cols_3.csv
  -> Saved: features_3.csv (13

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# --- 1. DETERMINING FILE PATHS ---
features_dir = "../02_Processed_Data/03_Calculations_Features"
chatter_csv_path = "../01_Raw_Data/Chatter.csv"

# Output directories (Separated by labels)
output_dir_0 = "../02_Processed_Data/04_Label_0"
output_dir_1 = "../02_Processed_Data/05_Label_1"

os.makedirs(output_dir_0, exist_ok=True)
os.makedirs(output_dir_1, exist_ok=True)

# --- 2. FILE-SPECIFIC CHATTER STARTING PERCENTAGES (%) ---
threshold_percentages = {
    'A': 0.45, 'B': 0.095, 'C': 0.24, 'D': 0.22, 'E': 0.31,
    'F': 0.39, 'G': 0.30, 'H': 0.38, 'I': 0.31, 
}

# --- 3. READING THE CHATTER FILE (LABEL LIST) ---
df_labels = pd.read_csv(chatter_csv_path, sep=';')

# Make the 'Experiment No' column a string, strip spaces, and CONVERT TO UPPERCASE
df_labels['Experiment No'] = df_labels['Experiment No'].astype(str).str.strip().str.upper()

# Create dictionary
label_dict = dict(zip(df_labels['Experiment No'], df_labels['Chatter']))

print(f"Label information of {len(label_dict)} experiments in total received from Chatter.csv.\n")

# --- 4. PROCESSING AND LABELING OF FEATURE FILES ---
file_paths = glob.glob(os.path.join(features_dir, '*.csv'))

if len(file_paths) == 0:
    print(f"❌ CRITICAL ERROR: No CSV files found in '{features_dir}'! Please check the file path.")
else:
    print(f"Checking a total of {len(file_paths)} files...\n")

processed_file_count = 0

for file_path in file_paths:
    file_name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    
    # --- ID EXTRACTION (Multi-Probability Check) ---
    # 1. Get from 'File_ID' or 'Dosya_ID' column if exists
    if 'File_ID' in df.columns:
        file_id = str(df['File_ID'].iloc[0]).strip().upper()
    elif 'Dosya_ID' in df.columns:
        file_id = str(df['Dosya_ID'].iloc[0]).strip().upper()
    else:
        # 2. Force extract from filename if not in datafram
        clean_name = file_name.replace('.csv', '').upper()
        if '-' in clean_name:
            file_id = clean_name.split('-')[-1].strip()
        else:
            file_id = clean_name.split('_')[-1].strip()
            
    # --- FINAL CLEANUP FOR ID MATCHING (THE FIX) ---
    # Remove unwanted prefixes to match Chatter.csv strictly
    file_id = file_id.replace('CUTTING_', '').replace('FEATURES_', '').replace('CUTTING', '')
    if '_' in file_id: # Fallback: if it's something like "DATA_11", grab the "11"
        file_id = file_id.split('_')[-1]
        
    # --- LABEL CHECK ---
    if file_id not in label_dict:
        print(f"⚠️ SKIPPED: Extracted ID '{file_id}' from '{file_name}' not found in Chatter list!")
        continue
        
    label = label_dict[file_id]
    
    # --- LABEL 0 OPERATIONS (NO CHATTER - Fill all rows with 0) ---
    if label == 0:
        df['chatter'] = 0  
        out_path = os.path.join(output_dir_0, file_name)
        df.to_csv(out_path, index=False)
        print(f"✅ Label 0 -> {file_name} (ID: {file_id}) successfully saved.")
        processed_file_count += 1
        
    # --- LABEL 1 OPERATIONS (CHATTER - Separate by percentage) ---
    elif label == 1:
        if file_id in threshold_percentages:
            threshold = threshold_percentages[file_id]
            
            # If the Percent_Complete value is >= threshold, print 1, else 0
            df['chatter'] = np.where(df['Percent_Complete'] >= threshold, 1, 0)
            
            out_path = os.path.join(output_dir_1, file_name)
            df.to_csv(out_path, index=False)
            print(f"🔥 Label 1 -> {file_name} (ID: {file_id}, Threshold: %{threshold*100}) successfully saved.")
            processed_file_count += 1
        else:
            print(f"❌ ERROR: {file_name} (ID: {file_id}) is marked as Chatter=1 but no threshold percentage is given! Skipped.")

print(f"\n🎉 Process Completed! A 'chatter' column was added to a total of {processed_file_count} files and they were separated into their directories.")

Label information of 20 experiments in total received from Chatter.csv.

Checking a total of 20 files...

✅ Label 0 -> features_11.csv (ID: 11) successfully saved.
✅ Label 0 -> features_10.csv (ID: 10) successfully saved.
✅ Label 0 -> features_8.csv (ID: 8) successfully saved.
✅ Label 0 -> features_9.csv (ID: 9) successfully saved.
🔥 Label 1 -> features_H.csv (ID: H, Threshold: %38.0) successfully saved.
🔥 Label 1 -> features_I.csv (ID: I, Threshold: %31.0) successfully saved.
🔥 Label 1 -> features_G.csv (ID: G, Threshold: %30.0) successfully saved.
✅ Label 0 -> features_1.csv (ID: 1) successfully saved.
🔥 Label 1 -> features_F.csv (ID: F, Threshold: %39.0) successfully saved.
🔥 Label 1 -> features_D.csv (ID: D, Threshold: %22.0) successfully saved.
✅ Label 0 -> features_2.csv (ID: 2) successfully saved.
✅ Label 0 -> features_3.csv (ID: 3) successfully saved.
🔥 Label 1 -> features_E.csv (ID: E, Threshold: %31.0) successfully saved.
✅ Label 0 -> features_7.csv (ID: 7) successfully saved